In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr

from config_utils import load_config
from model import SimpleModel
from preprocessing import create_2024_preprocessor_no_downscaling
from utils import (
    input_vars_from_config,
    ordered_dataset_for_config,
    plot_global_field,
    plot_north_pole_field,
    target_var_from_config,
)

SECONDS_PER_DAY = 3600 * 24

In [ ]:
def to_monthly(ds):
    year = ds.date.dt.year
    month = ds.date.dt.month
    ds = ds.assign_coords(year=("date", year.data), month=("date", month.data))
    return ds.set_index(date=("year", "month")).unstack("date")


def to_dates(ds):
    ds_stacked = ds.stack(date=("year", "month"))
    datetime_index = pd.to_datetime(
        {
            "year": ds_stacked.year.values,
            "month": ds_stacked.month.values,
            "day": 1,
        }
    )
    return (
        ds_stacked
        .drop_vars(['date', 'year', 'month'])
        .assign_coords(date=datetime_index)
        .sortby("date")
    )

def nn_pred(ds, model, preprocessor):
    year, month = ds.year.values, ds.month.values
    lon, lat = ds.longitude.values, ds.latitude.values
    target_var = target_var_from_config(config)

    ds_ordered_np = (
        ordered_dataset_for_config(ds, config)
        .to_dataarray()
        .transpose("year", "month", "latitude", "longitude", "variable")
        .to_numpy()
    )

    data_torch = torch.from_numpy(ds_ordered_np).float()
    model_outputs = model(data_torch[..., : model.input_dim]).detach()

    pred = (
        preprocessor.preprocessors[-1]
        .inverse_transform(
            xr.Dataset(
                data_vars={target_var: (("year", "month", "latitude", "longitude"), model_outputs)},
                coords={"year": year, "month": month, "longitude": lon, "latitude": lat},
            )
        )
        .to_dataarray()
        .squeeze(dim="variable", drop=True)
    )
    return pred


def dRnn(ds, anomaly, model, preprocessor, variables):
    ds_copies = []
    for var_list in variables:
        if not isinstance(var_list, list):
            var_list = [var_list]
        ds_copies.append(ds.assign({v: ds[v] + anomaly[v] for v in var_list}))

    ds_original = preprocessor.transform(ds)
    ds_perturbed = [preprocessor.transform(d) for d in ds_copies]

    pred_original = nn_pred(ds_original, model, preprocessor)
    return [(nn_pred(ds_p, model, preprocessor) - pred_original) / SECONDS_PER_DAY for ds_p in ds_perturbed]
    
def dRnn_cross(ds, anomaly, model, preprocessor, variable_pairs):
    """
    Compute second-order cross terms via interaction contrast:
        cross(i,j) = NN(X + dXi + dXj) - NN(X + dXi) - NN(X + dXj) + NN(X)

    variable_pairs: list of (var_i, var_j) tuples, where each var can be
                    a string or list of strings (matching dRnn convention).
    """
    # Flatten to get all unique variable groups needed
    all_var_lists = set()
    for vi, vj in variable_pairs:
        if not isinstance(vi, list): vi = [vi]
        if not isinstance(vj, list): vj = [vj]
        all_var_lists.add(tuple(vi))
        all_var_lists.add(tuple(vj))
        all_var_lists.add(tuple(vi + vj))

    ds_original = preprocessor.transform(ds)
    pred_original = nn_pred(ds_original, model, preprocessor)

    preds = {}
    for var_list in all_var_lists:
        ds_perturbed = preprocessor.transform(
            ds.assign({v: ds[v] + anomaly[v] for v in var_list})
        )
        preds[var_list] = (nn_pred(ds_perturbed, model, preprocessor))

    # Compute interaction contrasts
    results = []
    for vi, vj in variable_pairs:
        if not isinstance(vi, list): vi = [vi]
        if not isinstance(vj, list): vj = [vj]
        ki, kj, kij = tuple(vi), tuple(vj), tuple(vi + vj)
        results.append((preds[kij] - preds[ki] - preds[kj] + pred_original) / SECONDS_PER_DAY)

    return results

def global_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))
    weights.name = "weights"
    return da.weighted(weights).mean(dim=["latitude", "longitude"])

def global_date_series(da):
    da = to_dates(da.copy(deep=True))
    return global_mean(da).compute()

def setup_timeseries_plot():
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.set_xticks(pd.date_range(start='2007', end='2017', freq='YS', inclusive='both'), np.arange(2007, 2018))
    ax.set_xlabel('date')
    ax.grid(alpha=0.3)

    return fig, ax
def integrate_over_pressure_levels(sp, da):
    p = da.level * 100.0
    return da.where(p <= sp, 0).sum(dim="level")

In [ ]:
year, month = 2012, 9
output_dir = Path(f'closure_test_2/{year}_{month:02d}')
output_dir_clr = output_dir / 'clr'
output_dir_all = output_dir / 'all'

output_dir_clr.mkdir(exist_ok=True, parents=True)
output_dir_all.mkdir(exist_ok=True, parents=True)

(output_dir_clr / 'np').mkdir(exist_ok=True, parents=True)
(output_dir_all / 'np').mkdir(exist_ok=True, parents=True)

## Load Model And Data

In [ ]:
K_a = xr.open_dataset('data/ERA5_kernels/ERA5_kernel_alb_TOA.nc')
kernel_grid = {'latitude': K_a.latitude, 'longitude': K_a.longitude}

np_mask = K_a.latitude.values > 75

In [ ]:
data_path = Path('data/era5')
ds = xr.open_mfdataset(data_path.glob('era5_single_levels_monthly_*.nc'))
ds = ds.sel(date=slice('2007-01', '2016-12')).interp(**kernel_grid)
ds['tsr'] = ds.tsr / SECONDS_PER_DAY

ds_monthly = to_monthly(ds.copy(deep=True))
ds_monthly_means = ds_monthly.mean('year')
anomaly = ds_monthly - ds_monthly_means

ds_monthly_clr = ds_monthly.copy(deep=True)
ds_monthly_clr = ds_monthly_clr.assign({v: xr.zeros_like(ds_monthly[v]) for v in ['tcc', 'hcc', 'mcc', 'lcc', 'tciw', 'tclw']})

In [ ]:
ds_tsrc = xr.open_mfdataset(data_path.glob('era5_tsrc_monthly_*.nc'))
ds_tsrc = ds_tsrc.sel(date=slice('2007-01', '2016-12')).tsrc
ds_tsrc = ds_tsrc.interp(**kernel_grid).compute() / SECONDS_PER_DAY
ds_tsrc_monthly = to_monthly(ds_tsrc.copy(deep=True))
dR_clr = ds_tsrc_monthly - ds_tsrc_monthly.mean('year')

In [ ]:
dR = (anomaly.tsr.interp(**kernel_grid)).compute()

In [ ]:
config = load_config("configs/model/fal/sobolev_mar_june_sep_dec_2011_2014.yaml")
checkpoint_path = Path(config.train.checkpoint_dir) / "best_model.pt"
checkpoint_data = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model = SimpleModel(checkpoint_data["config"])
model.load_state_dict(checkpoint_data["model_state_dict"])
model.eval()

preprocessor = create_2024_preprocessor_no_downscaling(
    input_vars=input_vars_from_config(config),
    target_var=target_var_from_config(config),
    ecod=config.preprocess.ecod,
)
preprocessor.load(model.config.preprocess.params_dir)

## Plot Input Anomalies

In [ ]:
lat = K_a.latitude
lon = K_a.longitude
for var, label, vmax in zip(['fal', 'hcc', 'mcc', 'lcc', 'tcwv', 'tco3', 'tsr'], ['', '', '', '', 'kg/m^2', 'kg/m^2', 'W/m^2'], [0.5, None, None, None, None, None, 45]):
    field = anomaly.sel(month=month, year=year)[var].compute()
    title = r'$\Delta$' + f'{var}'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    if vmax == None:
        vmax = np.max(np.abs(field))

    print(var, np.max(np.abs(field)).values)
    plot_global_field    (field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label=label)
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}_np.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label=label, boundary=75)

In [ ]:
pp = preprocessor.transform(ds_monthly.sel(month=month,year=year))
ppm = preprocessor.transform(ds_monthly_means.sel(month=month))
ppa = pp - ppm
anomaly_ecod = ppa.ecod.compute()
print(var, np.max(np.abs(field)).values)

In [ ]:
var = 'ecod'
label = 'ecod'
field = anomaly_ecod
title = r'$\Delta$' + f'{var}'
ann = global_mean(field).values
ann_np = global_mean(field[np_mask]).values
vmax = np.max(np.abs(field))

plot_global_field    (field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label=label)
plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}_np.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label=label, boundary=75)

In [ ]:
var = 'tsrc'
label = 'tsrc'
field = dR_clr.sel(month=month, year=year)
title = r'$\Delta$' + f'{var}'
ann = global_mean(field).values
ann_np = global_mean(field[np_mask]).values
vmax = np.max(np.abs(field))

plot_global_field    (field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label=label)
plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir/ f'delta_{var}_np.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label=label, boundary=75)

## Neural Network Responses

In [ ]:
dR_components = dRnn(
    ds_monthly,
    anomaly,
    model,
    preprocessor,
    ['fal', ['tcc', 'hcc', 'mcc', 'lcc', 'tciw', 'tclw'], 'tcwv', 'tco3'],
)

dR_components_clr = dRnn(
    ds_monthly_clr,
    anomaly,
    model,
    preprocessor,
    ['fal', 'tcwv', 'tco3'],
)

dR_a_nn, dR_c_nn, dR_q_nn, dR_co3_nn = dR_components
dR_a_nn_clr, dR_q_nn_clr, dR_co3_nn_clr = dR_components_clr

In [ ]:
print(np.max(np.abs(dR_a_nn.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_c_nn.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_q_nn.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_co3_nn.sel(month=month, year=year) )).values)
print('-'*40)
print(np.max(np.abs(dR_a_nn_clr.sel(month=month, year=year) )).values)
print('-')
print(np.max(np.abs(dR_q_nn_clr.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_co3_nn_clr.sel(month=month, year=year) )).values)

vmax_a, vmax_c, vmax_q, vmax_co3 = 40, 60, 7, 1

In [ ]:
for dR, label, vmax in zip([dR_a_nn, dR_c_nn, dR_q_nn], ['a', 'c', 'q'], [vmax_a, vmax_c, vmax_q]):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{label}}}^{{NN}}$'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=output_dir_all / f'dR_{label}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_all / 'np' / f'dR_{label}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

In [ ]:
for dR, label, vmax in zip([dR_a_nn_clr, dR_q_nn_clr], ['a,clr', 'q,clr'], [60, 4]):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{label}}}^{{NN}}$'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=output_dir_clr / f'dR_{label}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_clr / 'np' / f'dR_{label}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

## Kernel Responses

In [ ]:
K_a = xr.open_dataset('data/ERA5_kernels/ERA5_kernel_alb_TOA.nc')
K_q = xr.open_dataset('data/ERA5_kernels/layer_specified_ta_wv_kernel/ERA5_kernel_wv_sw_nodp_TOA.nc')

In [ ]:
da = anomaly.fal.compute()
dR_a_k = K_a.TOA_all * da * 100
dR_a_k_clr = K_a.TOA_clr * da * 100

In [ ]:
ds_qt = xr.load_dataset('data/era5/era5_plev_qt_monthly_downscaled.nc')
"""
ds_qt = xr.open_mfdataset('data/era5/era5_plev_qt_monthly_*.nc')
#ds_qt = ds_qt.sel(date=slice('2007-01', '2016-12'))
#ds_qt = ds_qt.interp(latitude=K_q.latitude, longitude=K_q.longitude)
#ds_qt.to_netcdf('data/era5/era5_plev_qt_monthly_downscaled.nc')
"""

ds_qt_monthly = to_monthly(ds_qt.copy(deep=True))
ds_qt_anomaly = ds_qt_monthly - ds_qt_monthly.mean('year')

q = ds_qt_monthly.q
T = ds_qt_monthly.t
dq = ds_qt_anomaly.q

Rv = 461.52  # J / kg / K
Lv = 2260000  # J / kg
sp = ds_monthly.sp

# dT from dq using Clausius-Clapeyron approximation.
dT = (dq / q) * (Rv / Lv) * (T**2)

dR_q_k = integrate_over_pressure_levels(sp, K_q.TOA_all * dT).compute()
dR_q_k_clr = integrate_over_pressure_levels(sp, K_q.TOA_clr * dT).compute()

In [ ]:
#all_sky_residual_k = dR_k - dR_a_k - dR_q_k
dR_c_k = (dR - dR_clr) - (dR_a_k - dR_a_k_clr) - (dR_q_k - dR_q_k_clr)

In [ ]:
print(np.max(np.abs(dR_a_k.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_c_k.sel(month=month, year=year) )).values)
print(np.max(np.abs(dR_q_k.sel(month=month, year=year) )).values)
print('-'*40)
print(np.max(np.abs(dR_a_k_clr.sel(month=month, year=year) )).values)
print('-')
print(np.max(np.abs(dR_q_k_clr.sel(month=month, year=year) )).values)


In [ ]:
for dR, label, vmax in zip([dR_a_k, dR_q_k, dR_c_k], ['a', 'q', 'c'], [vmax_a, vmax_q, vmax_c]):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{label}}}^{{K}}$'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=output_dir_all / f'k_dR_{label}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_all / 'np' / f'k_dR_{label}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

for dR, label, vmax in zip([dR_a_k_clr, dR_q_k_clr], ['a,clr', 'q,clr'], [vmax_a, vmax_q]):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{label}}}^{{K}}$'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=output_dir_clr / f'k_dR_{label}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_clr / 'np' / f'k_dR_{label}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

## NN vs Kernel Anomaly

In [ ]:
dR_sum_nn = dR_a_nn + dR_c_nn + dR_q_nn
dR_sum_k = dR_a_k + dR_c_k + dR_q_k

dR_sum_nn_clr = dR_a_nn_clr + dR_q_nn_clr
dR_sum_k_clr = dR_a_k_clr + dR_q_k_clr

dR_res_nn = dR - dR_sum_nn
dR_res_k = dR - dR_sum_k

dR_res_nn_clr = dR_clr - dR_sum_nn
dR_res_k_clr = dR_clr - dR_sum_k

In [ ]:
print(np.max(np.abs(dR_sum_nn).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_sum_k).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_sum_nn_clr).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_sum_k_clr).sel(year=year,month=month)).values)
###
print(np.max(np.abs(dR_res_nn).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_res_k).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_res_nn_clr).sel(year=year,month=month)).values)
print(np.max(np.abs(dR_res_k_clr).sel(year=year,month=month)).values)

In [ ]:
for dR, sub, sup, path, prefix, vmax in zip(
    [dR_sum_nn, dR_sum_k, dR_sum_nn_clr, dR_sum_k_clr, dR_res_nn, dR_res_k, dR_res_nn_clr, dR_res_k_clr],
    ['sum'] * 2 + ['sum,clr'] * 2 + ['res'] * 2  + ['res,clr'] * 2,
    ['NN', 'K'] * 4,
    ([output_dir_all] * 2 + [output_dir_clr] * 2) * 2,
    ['', 'k_'] * 4,
    [55, 55, 60, 60, 55, 55, 90, 90]
):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{sub}}}^{{{sup}}}$'
    
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=path / (prefix + f'dR_{sub}.png'),    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=path / 'np' / (prefix + f'dR_{sub}.png'), vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

## Neural Network Vs Kernel Timeseries

In [ ]:
series = {
    'dR_a_nn': global_date_series(dR_a_nn),
    'dR_a_k': global_date_series(dR_a_k),
    'dR_q_nn': global_date_series(dR_q_nn),
    'dR_q_k': global_date_series(dR_q_k),
    'dR_c_nn': global_date_series(dR_c_nn),
    'dR_c_k': global_date_series(dR_c_k),

    'dR_a_nn_clr': global_date_series(dR_a_nn_clr),
    'dR_a_k_clr': global_date_series(dR_a_k_clr),
    'dR_q_nn_clr': global_date_series(dR_q_nn_clr),
    'dR_q_k_clr': global_date_series(dR_q_k_clr),
}

series_np = {
    'dR_a_nn': global_date_series(dR_a_nn.isel(latitude=np_mask)),
    'dR_a_k': global_date_series(dR_a_k.isel(latitude=np_mask)),
    'dR_q_nn': global_date_series(dR_q_nn.isel(latitude=np_mask)),
    'dR_q_k': global_date_series(dR_q_k.isel(latitude=np_mask)),
    'dR_c_nn': global_date_series(dR_c_nn.isel(latitude=np_mask)),
    'dR_c_k': global_date_series(dR_c_k.isel(latitude=np_mask)),

    'dR_a_nn_clr': global_date_series(dR_a_nn_clr.isel(latitude=np_mask)),
    'dR_a_k_clr': global_date_series(dR_a_k_clr.isel(latitude=np_mask)),
    'dR_q_nn_clr': global_date_series(dR_q_nn_clr.isel(latitude=np_mask)),
    'dR_q_k_clr': global_date_series(dR_q_k_clr.isel(latitude=np_mask)),
}


for var in ['a', 'q', 'c']:
    fig, ax = setup_timeseries_plot()
    ax.axhline(0, alpha=0.1)
    ylabel = rf'$\Delta R_{{{var}}}$ ($W m^2$)'
    
    for source in ['nn', 'k']:
        name = f'dR_{var}_{source}'
        ax.scatter(x=series[name]['date'], y=series[name], s=1, label=source)

    ax.set_yticks(np.arange(-2,2.5,0.5))
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir_all / f'timeseries_{var}.png', dpi=200)
    plt.close(fig)
    
for var in ['a', 'q']:
    fig, ax = setup_timeseries_plot()
    ax.axhline(0, alpha=0.1)
    ylabel = rf'$\Delta R_{{{var},clr}}$ ($W /m^2$)'

    ticks = [-2, 0, 2]
    for source in ['nn', 'k']:
        name = f'dR_{var}_{source}_clr'
        ax.scatter(x=series[name]['date'], y=series[name], s=1, label=source)
        
    ax.set_yticks(np.arange(-2,2.5,0.5))
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir_clr / f'timeseries_{var}.png', dpi=200)
    plt.close(fig)

for var in ['a', 'q', 'c']:
    fig, ax = setup_timeseries_plot()
    ax.axhline(0, alpha=0.1)
    ylabel = rf'$\Delta R_{{{var}}}$ ($W m^2$)'
    
    for source in ['nn', 'k']:
        name = f'dR_{var}_{source}'
        ax.scatter(x=series_np[name]['date'], y=series[name], s=1, label=source)

    ax.set_yticks(np.arange(-2,2.5,0.5))
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir_all / 'np' / f'timeseries_{var}.png', dpi=200)
    plt.close(fig)
for var in ['a', 'q']:
    fig, ax = setup_timeseries_plot()
    ax.axhline(0, alpha=0.1)
    ylabel = rf'$\Delta R_{{{var},clr}}$ ($W /m^2$)'

    ticks = [-2, 0, 2]
    for source in ['nn', 'k']:
        name = f'dR_{var}_{source}_clr'
        ax.scatter(x=series_np[name]['date'], y=series[name], s=1, label=source)
        
    ax.set_yticks(np.arange(-2,2.5,0.5))
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir_clr / 'np' / f'timeseries_{var}.png', dpi=200)
    plt.close(fig)


In [ ]:
series = {
    'dR_res_nn': np.sqrt(global_date_series(np.power(dR_res_nn, 2))),
    'dR_res_k': np.sqrt(global_date_series(np.power(dR_res_k, 2))),
    'dR_res_nn_clr': np.sqrt(global_date_series(np.power(dR_res_nn_clr, 2))),
    'dR_res_k_clr': np.sqrt(global_date_series(np.power(dR_res_k_clr, 2))),
}

series_np = {
    'dR_res_nn': np.sqrt(global_date_series(np.power(dR_res_nn.isel(latitude=np_mask), 2))),
    'dR_res_k': np.sqrt(global_date_series(np.power(dR_res_k.isel(latitude=np_mask), 2))),
    'dR_res_nn_clr': np.sqrt(global_date_series(np.power(dR_res_nn_clr.isel(latitude=np_mask), 2))),
    'dR_res_k_clr': np.sqrt(global_date_series(np.power(dR_res_k_clr.isel(latitude=np_mask), 2))),
}

for clr in [False, True]:
    fig, ax = setup_timeseries_plot()
    ylabel = rf'RMSE $\Delta R_{{net,clr}}$ ($W m^{{-2}}$)' if clr else rf'RMSE $\Delta R_{{net}}$ ($W m^{{-2}}$)'
    save_path = output_dir_clr if clr else output_dir_all

    for source in ['nn', 'k']:
        name = f'dR_res_{source}' + ('_clr' if clr else '')
        ax.scatter(x=series[name]['date'], y=series[name], s=1, label=source)

    ax.set_yticks(np.arange(0,21,4))
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.5)
    fig.tight_layout()
    fig.savefig(save_path / f'timeseries_rmse.png', dpi=200)
    plt.close(fig)

for clr in [False, True]:
    fig, ax = setup_timeseries_plot()
    ylabel = rf'RMSE $\Delta R_{{net,clr}}$ ($W m^{{-2}}$)' if clr else rf'RMSE $\Delta R_{{net}}$ ($W m^{{-2}}$)'
    save_path = output_dir_clr if clr else output_dir_all

    for source in ['nn', 'k']:
        name = f'dR_res_{source}' + ('_clr' if clr else '')
        ax.scatter(x=series_np[name]['date'], y=series_np[name], s=1, label=source)
        
    ax.set_yticks(np.arange(0,21,4))
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.5)
    fig.tight_layout()
    fig.savefig(save_path / 'np' / f'timeseries_rmse.png', dpi=200)
    plt.close(fig)

In [ ]:
net_series = {
    'nn': global_date_series(dR_sum_nn),
    'kernel': global_date_series(dR_sum_k),
    'era5': global_date_series(dR),
}

net_series_clr = {
    'nn': global_date_series(dR_sum_nn_clr),
    'kernel': global_date_series(dR_sum_k_clr),
    'era5': global_date_series(dR_clr),
}

net_series_np = {
    'nn': global_date_series(dR_sum_nn.isel(latitude=np_mask)),
    'kernel': global_date_series(dR_sum_k.isel(latitude=np_mask)),
    'era5': global_date_series(dR.isel(latitude=np_mask)),
}

net_series_np_clr = {
    'nn': global_date_series(dR_sum_nn_clr.isel(latitude=np_mask)),
    'kernel': global_date_series(dR_sum_k_clr.isel(latitude=np_mask)),
    'era5': global_date_series(dR_clr.isel(latitude=np_mask)),
}

fig, ax = setup_timeseries_plot()
ax.axhline(0, alpha=0.1)
for name in ['nn', 'kernel', 'era5']:
    ax.scatter(x=net_series[name]['date'], y=net_series[name], s=1, label=name)
ax.set_yticks(np.arange(-2,2.5,0.5))
ax.set_ylabel('$\Delta R$ (W/m$^2$)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir_all / 'timeseries_net.png', dpi=200)
plt.show()

fig, ax = setup_timeseries_plot()
ax.axhline(0, alpha=0.1)
for name in ['nn', 'kernel', 'era5']:
    ax.scatter(x=net_series_clr[name]['date'], y=net_series_clr[name], s=1, label=name)
ax.set_yticks(np.arange(-2,2.5,0.5))
ax.set_ylabel('$\Delta R$ (W/m$^2$)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir_clr / 'timeseries_net.png', dpi=200)
plt.show()

fig, ax = setup_timeseries_plot()
ax.axhline(0, alpha=0.1)
for name in ['nn', 'kernel', 'era5']:
    ax.scatter(x=net_series_np[name]['date'], y=net_series_np[name], s=1, label=name)
ax.set_yticks(np.arange(-2,2.5,0.5))
ax.set_ylabel('$\Delta R$ (W/m$^2$)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir_all / 'np' / 'timeseries_net.png', dpi=200)
plt.show()

fig, ax = setup_timeseries_plot()
ax.axhline(0, alpha=0.1)
for name in ['nn', 'kernel', 'era5']:
    ax.scatter(x=net_series_np_clr[name]['date'], y=net_series_np_clr[name], s=1, label=name)
ax.set_yticks(np.arange(-2,2.5,0.5))
ax.set_ylabel('$\Delta R$ (W/m$^2$)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir_clr / 'np' / 'timeseries_net.png', dpi=200)
plt.show()

## Cross terms

In [ ]:
variable_pairs = [
    ('fal', 'tcwv'),
    ('fal', ['hcc', 'mcc', 'lcc', 'tcc', 'tciw', 'tclw']),
    ('tcwv', ['hcc', 'mcc', 'lcc', 'tcc', 'tciw', 'tclw']),
]

dR_nn_cross = dRnn_cross(ds_monthly, anomaly, model, preprocessor, variable_pairs)

dR_aq_nn, dR_ac_nn, dR_qc_nn = dR_nn_cross

In [ ]:
pair_labels = [
    ('a', 'q'),
    ('a', 'c'),
    ('q', 'c'),
]
vmax_cross = [
    float(np.abs(r.sel(month=month, year=year)).max().compute())
    for r in dR_nn_cross
]

for dR, (li, lj), vmax in zip(dR_nn_cross, pair_labels, vmax_cross):
    field = dR.sel(month=month, year=year) 
    title = fr'$\Delta R_{{{li},{lj}}}^{{NN}}$'
    ann = global_mean(field).values
    ann_np = global_mean(field[np_mask]).values

    plot_global_field    (field, lon, lat, title=title, save_path=output_dir_all / f'cross_dR_{li},{lj}.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
    plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_all / 'np' / f'cross_dR_{li},{lj}.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2', boundary=75)

In [ ]:
dR_aq_nn, dR_ac_nn, dR_qc_nn = dR_nn_cross
dR_sum_nn_cross = dR_sum_nn + dR_aq_nn + dR_ac_nn + dR_qc_nn

dR_res_nn_cross = dR - dR_sum_nn_cross


In [ ]:
print(np.max(np.abs(dR_sum_nn_cross.sel(month=month, year=year))).values)
print(np.max(np.abs(dR_res_nn_cross.sel(month=month, year=year))).values)

In [ ]:
field = dR_sum_nn_cross.sel(month=month, year=year) 
title = fr'$\Delta R_{{sum,cross}}^{{NN}}$'
ann = global_mean(field).values
ann_np = global_mean(field[np_mask]).values
vmax = 55

plot_global_field    (field, lon, lat, title=title, save_path=output_dir_all / f'cross_dR_sum.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_all / 'np' / f'cross_dR_sum.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2')

field = dR_res_nn_cross.sel(month=month, year=year) 
title = fr'$\Delta R_{{res,cross}}^{{NN}}$'
ann = global_mean(field).values
ann_np = global_mean(field[np_mask]).values
vmax = 55

plot_global_field    (field, lon, lat, title=title, save_path=output_dir_all / f'cross_dR_res.png',    vmin=-vmax, vmax=vmax, annotation=f'{ann:.2f}', label='W/m^2')
plot_north_pole_field(field, lon, lat, title=title, save_path=output_dir_all / 'np' / f'cross_dR_res.png', vmin=-vmax, vmax=vmax, annotation=f'{ann_np:.2f}', label='W/m^2')

In [ ]:
net_series = {
    'era5': global_date_series(dR),
    'nn': global_date_series(dR_sum_nn),
    'kernel': global_date_series(dR_sum_k),
    'nn_cross': global_date_series(dR_sum_nn_cross),
    'cross': global_date_series(dR_aq_nn + dR_ac_nn + dR_qc_nn),
}

fig, ax = plt.subplots(figsize=(10, 3))
ax.axhline(0, alpha=0.1)
colors = ['tab:green', 'tab:blue', 'tab:orange', 'purple', 'tab:red']
for i, (name, s) in enumerate(net_series.items()):
    if name == 'era5':
        size = 3
    else:
        size = 1
    color = colors[i]
    ax.scatter(x=s['date'], y=s, s=size, label=name, color=color)
ax.set_yticks([-2, 0, 2])
ax.set_xticks(
    pd.date_range(start='2007', end='2017', freq='YS', inclusive='both'),
    np.arange(2007, 2018)
)
ax.set_ylabel('W/m^2')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir_all / 'timeseries_sum_cross.png', dpi=200)
plt.show()

In [ ]:
def weighted_residuals_by_month(da, n_samples=2000):
    result = []
    for m in range(1, 13):
        field = da.sel(month=m).compute().values  # could be (lat, lon, year) or (year, lat, lon)

        # transpose to (year, lat, lon)
        dims = list(da.sel(month=m).dims)
        field = np.moveaxis(field, dims.index('year'), 0)
        field = np.moveaxis(field, dims.index('latitude') if dims.index('latitude') < dims.index('year') else dims.index('latitude') - 1 + 1, 1)
        # simpler: just use xarray transpose before .values
        field = da.sel(month=m).transpose('year', 'latitude', 'longitude').compute().values
        n_year, n_lat, n_lon = field.shape

        lat_vals = da.latitude.values
        weights = np.clip(np.cos(np.deg2rad(lat_vals)), 0, None)
        w = weights[None, :, None] * np.ones((n_year, n_lat, n_lon))

        flat_res = field.flatten()
        p = w.flatten().copy()
        valid = np.isfinite(flat_res)
        p[~valid] = 0.0
        p = p / p.sum()

        indices = np.random.choice(len(flat_res), size=n_samples, p=p)
        result.append(flat_res[indices])
    return result

In [ ]:
residuals = {
    'nn':       weighted_residuals_by_month(dR_res_nn),
    'nn_cross': weighted_residuals_by_month(dR_res_nn_cross),
    'kernel':   weighted_residuals_by_month(dR_res_k),
}

residuals_np = {
    'nn':       weighted_residuals_by_month(dR_res_nn.isel(latitude=np_mask)),
    'nn_cross': weighted_residuals_by_month(dR_res_nn_cross.isel(latitude=np_mask)),
    'kernel':   weighted_residuals_by_month(dR_res_k.isel(latitude=np_mask)),
}

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
colors = {'nn': 'tab:blue', 'nn_cross': 'purple', 'kernel': 'tab:orange'}
width = 0.25
offsets = {'nn': -width, 'nn_cross': 0, 'kernel': width}
months = np.arange(1, 13)

for title, data_dict, save_name in [
    ('Global', residuals, 'violin_residuals_by_month_global.png'),
    ('North Pole (>75°N)', residuals_np, 'violin_residuals_by_month_np.png'),
]:
    fig, ax = plt.subplots(figsize=(14, 4))

    for key in ['nn', 'nn_cross', 'kernel']:
        positions = months + offsets[key]
        parts = ax.violinplot(
            data_dict[key],
            positions=positions,
            widths=width * 0.9,
            showmedians=True,
            showextrema=False,
        )
        for pc in parts['bodies']:
            pc.set_facecolor(colors[key])
            pc.set_alpha(0.6)
        parts['cmedians'].set_color(colors[key])
        ax.scatter([], [], color=colors[key], label=key, s=20)

    ax.axhline(0, color='black', alpha=0.2, linewidth=0.8)
    ax.set_xticks(months)
    ax.set_xticklabels(month_names)
    ax.set_ylabel(r'$\Delta R_{\rm res}$ (W m$^{-2}$)')
    ax.set_title(title)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_dir_all / save_name, dpi=200)
    plt.show()